<a href="https://colab.research.google.com/github/Akhila-010/GenAI_Tasks/blob/main/GraphRAG_College_Data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install openai faiss-cpu PyPDF2 networkx

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 36.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 9.9 MB/s eta 0:00:00


In [ ]:
from openai import OpenAI
from google.colab import userdata

OPENAI_API_KEY=userdata.get('OPENAI_API_KEY')
client = OpenAI(api_key=OPENAI_API_KEY)

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving CSEDept.MinorinDataScienceCourseStructureandSyllabus.pdf to CSEDept.MinorinDataScienceCourseStructureandSyllabus.pdf
Saving Placement_report.pdf to Placement_report.pdf
Saving information-brochure_Delhi.pdf to information-brochure_Delhi (1).pdf


In [ ]:
import PyPDF2

def extract_text(files):
    text = ""
    for file in files:
        reader = PyPDF2.PdfReader(file)
        for page in reader.pages:
            content = page.extract_text()
            if content:
                text += content + "\n"
    return text

text = extract_text(uploaded)

In [ ]:
def chunk_text(text, size=500, overlap=100):
    chunks = []
    for i in range(0, len(text), size - overlap):
        chunks.append(text[i:i+size])
    return chunks

chunks = chunk_text(text)

In [ ]:
def get_embeddings(chunks):
    embeddings = []
    for chunk in chunks:
        res = client.embeddings.create(
            model="text-embedding-3-small",
            input=chunk
        )
        embeddings.append(res.data[0].embedding)
    return embeddings

embeddings = get_embeddings(chunks)

In [ ]:
import faiss
import numpy as np

embeddings_np = np.array(embeddings).astype("float32")
index = faiss.IndexFlatL2(len(embeddings_np[0]))
index.add(embeddings_np)

In [ ]:
def rag_search(query, k=5):
    q_embed = client.embeddings.create(
        model="text-embedding-3-small",
        input=query
    ).data[0].embedding

    D, I = index.search(np.array([q_embed]).astype("float32"), k)
    return [chunks[i] for i in I[0]]

In [ ]:
def extract_entities(text):
    res = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "Extract key entities like courses, subjects, companies, fees."},
            {"role": "user", "content": text[:3000]}
        ]
    )

    entities = res.choices[0].message.content.split(",")
    return [e.strip() for e in entities]

In [ ]:
entities = extract_entities(text)

In [ ]:
import networkx as nx

graph = nx.Graph()

def build_graph(entities):
    for i in range(len(entities)-1):
        graph.add_node(entities[i])
        graph.add_edge(entities[i], entities[i+1])

build_graph(entities)

In [ ]:
def graph_search(query):
    results = []

    for node in graph.nodes:
        if query.lower() in node.lower():
            results.append(node)
            results.extend(list(graph.neighbors(node)))

    return results

In [ ]:
def ask(query):

    # RAG context
    rag_context = "\n".join(rag_search(query))

    # Graph context
    graph_context = ", ".join(graph_search(query))

    final_context = f"""
    RAG Context:
    {rag_context}

    Graph Context:
    {graph_context}
    """

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "Answer using both context sources. If not found, say 'Not in documents'."},
            {"role": "user", "content": f"{final_context}\n\nQuestion: {query}"}
        ]
    )

    return response.choices[0].message.content

In [ ]:
print("Hybrid College AI Assistant Ready!")

while True:
    q = input("\nYou: ")

    if q.lower() == "exit":
        print("Chat ended.")
        break

    print("\nBot:", ask(q))

Hybrid College AI Assistant Ready!

You: what is the syllabus

Bot: The syllabus mentioned in the context pertains to Microeconomics at a 10+2+3 level and includes the following topics:

1. **Consumer Theory**: 
   - Preference
   - Utility and representation theorem
   - Budget constraint
   - Choice
   - Demand (ordinary and compensated)
   - Slutsky equations
   - Choice under risk and uncertainty
   - Revealed preference axioms

2. **Production and Costs with Perfectly Competitive Markets**: 
   - Technology
   - Isoquants
   - Production with one and more variable inputs
   - Returns to scale
   - Short run and long run analysis

Note: The questions in the examination may not be limited to these topics.

You: what type of courses college is offering ?

Bot: The college is offering various degree programs that require students to complete a minimum number of credits, which vary based on the program. The admissions are available for candidates holding degrees such as B. Tech., M.Sc.